In [1]:
import os
os.chdir("/root/EdgeTRM")
print(os.getcwd())

!git config --global --add safe.directory /root/EdgeTRM
# !rm -rf /root/EdgeTRM/EdgeTRM
 


/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f


In [2]:
# !git fetch origin
# !git reset --hard origin/main
!git pull origin main

From https://github.com/Seqaeon/EdgeTRM
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
# ── MUST RUN FIRST — fix duplicate trm.py on Modal volume ────────────────────
# There are two identical trm.py files on the Modal volume:
#   TinyRecursiveModels/trm.py
#   TinyRecursiveModels/models/recursive_reasoning/trm.py
#
# Python loads them as separate module objects, so patching one class
# has no effect on instances created from the other.
#
# Fix: replace the top-level copy with a symlink so both import paths
# resolve to the same file and the same Python module object.
#
# Run this cell ONCE per kernel, then restart the kernel.

import os, sys

trm_root = None
for path in sys.path:
    candidate = os.path.join(path, "trm.py")
    if os.path.exists(candidate) and "TinyRecursiveModels" in candidate:
        trm_root = candidate
        break

# Also check the known Modal volume path directly
modal_top = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/trm.py"
modal_sub = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/models/recursive_reasoning/trm.py"

for top, sub in [(modal_top, modal_sub)]:
    if not os.path.exists(sub):
        print(f"[SKIP] {sub} not found")
        continue
    if os.path.islink(top):
        print(f"✓ Already a symlink: {top} → {os.readlink(top)}")
    elif os.path.exists(top):
        os.rename(top, top + ".bak")
        os.symlink(sub, top)
        print(f"✓ Replaced {top} with symlink → {sub}")
        print("  Restart the kernel now, then run all cells from the top.")
    else:
        print(f"[SKIP] {top} not found")


✓ Already a symlink: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/trm.py → models/recursive_reasoning/trm.py


In [4]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

repo_root: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f
trm_root: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [5]:

!uv pip install --system {trm_root}
%uv pip install einops

Using Python 3.12.6 environment at: /usr/local
Resolved 64 packages in 1.95s
Building antlr4-python3-runtime==4.9.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
nvidia-nvjitlink-cu12 ------------------------------ 16.00 KiB/

In [6]:
import os
os.chdir('/root/EdgeTRM/TinyRecursiveModels')


In [7]:
import torch
import yaml
from trm import TinyRecursiveReasoningModel_ACTV1

def load_arc_model(checkpoint_path, config_text):
    # 1. Parse the YAML provided in your prompt
    raw_config = yaml.safe_load(config_text)
    
    # 2. Manually map and extract the required fields
    arch = raw_config['arch']
    
    # Construct the exact dictionary required by TinyRecursiveReasoningModel_ACTV1Config
    final_config = {
        "batch_size": 32,
        "seq_len": 900,
        "num_puzzle_identifiers": 225972, # Set to correct vocab size to prevent clamping
        "vocab_size": 12,
        "H_cycles": arch['H_cycles'],
        "L_cycles": arch['L_cycles'],
        "H_layers": arch['H_layers'],
        "L_layers": arch['L_layers'],
        "hidden_size": arch['hidden_size'],
        "expansion": arch['expansion'],
        "num_heads": arch['num_heads'],
        "pos_encodings": arch['pos_encodings'],
        "halt_max_steps": arch['halt_max_steps'],
        "halt_exploration_prob": arch['halt_exploration_prob'],
        "forward_dtype": arch.get('forward_dtype', 'bfloat16'),
        "mlp_t": arch.get('mlp_t', False),
        "puzzle_emb_ndim": arch.get('puzzle_emb_ndim', 512),
        "puzzle_emb_len": arch.get('puzzle_emb_len', 16),
        "no_ACT_continue": arch.get('no_ACT_continue', True)
    }

    # 1. Initialize the model
    model = TinyRecursiveReasoningModel_ACTV1(config_dict=final_config)
    
    # 2. Load the raw state_dict
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    
    # 3. If the state_dict is nested under a 'model' key
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    # 4. Strip prefix and load cleaned state_dict
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            clean_state_dict[k[len(unwanted_prefix):]] = v
        else:
            clean_state_dict[k] = v
            
    # 5. Robust resizing of the puzzle embedding weights to preserve learned parameters
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = model.inner.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
            mean_emb = torch.mean(puzzle_emb, dim=0)
            new_weights[:] = mean_emb
            min_rows = min(puzzle_emb.shape[0], expected_shape[0])
            new_weights[:min_rows] = puzzle_emb[:min_rows]
            clean_state_dict[puzzle_emb_name] = new_weights
            
    # 6. Load the state dict
    model.load_state_dict(clean_state_dict)
    model.__dict__['model'] = model
    model.eval()
    print("Prefixes stripped and model loaded successfully!")
    return model
config_data = """
arch:
  H_cycles: 3
  H_layers: 0
  L_cycles: 4
  L_layers: 2
  expansion: 4
  forward_dtype: bfloat16
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_len: 16
  puzzle_emb_ndim: 512
global_batch_size: 512
"""

checkpoint = "eval_checkpoint/step_14907" # Ensure this file is in your directory

try:
    model = load_arc_model(checkpoint, config_data)
    print("Model successfully loaded!")
except Exception as e:
    print(f"Error: {e}")

model

Prefixes stripped and model loaded successfully!
Model successfully loaded!


TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)

In [10]:
# # Optional: Shrink the embedding table to exactly fit the active dataset vocabulary
# import torch
# from torch import nn
# def get_inner(m):
#     """Unwrap DataParallel / compiled model to get inner TRM."""
#     m2 = m.module if hasattr(m, 'module') else m
#     return m2._orig_mod if hasattr(m2, '_orig_mod') else m2
# with torch.no_grad():
#     active_vocab_size = 225972
#     inner_model = get_inner(model)
    
#     # Slice the weight table to only keep the active range
#     shrunk_weights = inner_model.puzzle_emb.weights[:active_vocab_size].clone()
#     inner_model.puzzle_emb.weights = nn.Buffer(shrunk_weights, persistent=True)
    
#     # Update config size
#     inner_model.config.num_puzzle_identifiers = active_vocab_size

# print(f"Squeezed embedding weights table to: {inner_model.puzzle_emb.weights.shape}")


Squeezed embedding weights table to: torch.Size([225972, 512])


In [11]:
# ── Cell 9: Sparsity Audit ────────────────────────────────────────────────────
def count_zero_params(m):
    total, zeros = 0, 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros, total

for name, m in [('Original', model)]:
    z, t = count_zero_params(m)
    print(f'{name:<15} zero={z:,}/{t:,}  ({100*z/t:.1f}%)')

Original        zero=512/6,829,058  (0.0%)


In [12]:
%%writefile eval-arc.py

from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil
import copy

import torch
import torch.distributed as dist
from torch import nn
from torch.utils.data import DataLoader

import tqdm
#import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from adam_atan2_pytorch import AdamAtan2

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed
from models.ema import EMAHelper


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str
    loss: LossConfig


class EvaluatorConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra="allow")
    name: str


class PretrainConfig(pydantic.BaseModel):
    # Config
    arch: ArchConfig
    # Data
    data_paths: List[str]
    data_paths_test: List[str] = []
    # Evaluators
    evaluators: List[EvaluatorConfig] = []

    # Hyperparams
    global_batch_size: int
    epochs: int

    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int

    weight_decay: float
    beta1: float
    beta2: float

    # Puzzle embedding
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float

    # Names
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    load_checkpoint: Optional[str] = None
    checkpoint_path: Optional[str] = None

    # Extras
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    min_eval_interval: Optional[int] = 0 # when to start eval
    eval_save_outputs: List[str] = []

    ema: bool = False # use Exponential-Moving-Average
    ema_rate: float = 0.999 # EMA-rate
    freeze_weights: bool = False # If True, freeze weights and only learn the embeddings

@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any

    step: int
    total_steps: int


def create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed,
        dataset_paths=config.data_paths_test if len(config.data_paths_test)>0 and split=="test" else config.data_paths,
        rank=rank,
        num_replicas=world_size,
        **kwargs
    ), split=split)
    dataloader = DataLoader(
        dataset,
        batch_size=None,
        num_workers=1,
        prefetch_factor=8,
        pin_memory=True,
        persistent_workers=True
    )
    return dataloader, dataset.metadata


def create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,  # type: ignore
        batch_size=config.global_batch_size // world_size,
        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,
        causal=False  # Non-autoregressive
    )

    # Instantiate model with loss head
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    with torch.device("cuda"):
        model: nn.Module = model_cls(model_cfg)
        print(model)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model)  # type: ignore

        # Load checkpoint
        if rank == 0:
            load_checkpoint(model, config)

        # Broadcast parameters from rank 0
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    # Optimizers and lr
    if config.arch.puzzle_emb_ndim == 0:
        optimizers = [
            AdamAtan2(
                model.parameters(),
                lr=0.0001,  # Needs to be set by scheduler
                weight_decay=config.weight_decay,
                betas=(config.beta1, config.beta2)
            )
        ]
        optimizer_lrs = [
            config.lr
        ]
    elif config.freeze_weights:
        optimizers = [
            CastedSparseEmbeddingSignSGD_Distributed(
                model.model.puzzle_emb.buffers(),  # type: ignore
                lr=0,  # Needs to be set by scheduler
                weight_decay=config.puzzle_emb_weight_decay,
                world_size=world_size
            )
        ]
        optimizer_lrs = [
            config.puzzle_emb_lr
        ]
    else:
        optimizers = [
            CastedSparseEmbeddingSignSGD_Distributed(
                model.model.puzzle_emb.buffers(),  # type: ignore
                lr=0,  # Needs to be set by scheduler
                weight_decay=config.puzzle_emb_weight_decay,
                world_size=world_size
            ),
            AdamAtan2(
                model.parameters(),
                lr=0.0001,  # Needs to be set by scheduler
                weight_decay=config.weight_decay,
                betas=(config.beta1, config.beta2)
            )
        ]
        optimizer_lrs = [
            config.puzzle_emb_lr,
            config.lr
        ]

    return model, optimizers, optimizer_lrs

def mix_weights_direct(device, alpha, net, nets):
    sd = []
    for i in range(len(nets)):
        sd += [nets[i].state_dict()]
    sd_alpha = {}
    for k in sd[0].keys():
        comb_net = alpha[0]*sd[0][k].to(device)
        for i in range(1,len(nets)):
            comb_net += alpha[i]*sd[i][k].to(device)
        sd_alpha[k] =  comb_net
    net.load_state_dict(sd_alpha)
    return net

def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))

    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    # Estimated total training steps
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)

    # Model
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, rank=rank, world_size=world_size)

    return TrainState(
        step=0,
        total_steps=total_steps,

        model=model,
        optimizers=optimizers,
        optimizer_lrs=optimizer_lrs,
        carry=None
    )


def save_train_state(config: PretrainConfig, train_state: TrainState):
    # FIXME: Only saved model.
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))


def load_checkpoint(model: nn.Module, config: PretrainConfig):
    if config.load_checkpoint is not None:
        print(f"Loading checkpoint {config.load_checkpoint}")

        # Load state dict
        state_dict = torch.load(config.load_checkpoint, map_location="cuda")

        # Resize and reset puzzle emb if needed
        puzzle_emb_name = "_orig_mod.model.inner.puzzle_emb.weights"
        expected_shape: torch.Size = model.model.puzzle_emb.weights.shape  # type: ignore
        if puzzle_emb_name in state_dict:
            puzzle_emb = state_dict[puzzle_emb_name]
            if puzzle_emb.shape != expected_shape:
                print(f"Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
                # Create new tensor with expected shape, initialized to mean
                new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
                mean_emb = torch.mean(puzzle_emb, dim=0)
                new_weights[:] = mean_emb
                
                # Copy all overlapping rows to preserve learned weights
                min_rows = min(puzzle_emb.shape[0], expected_shape[0])
                new_weights[:min_rows] = puzzle_emb[:min_rows]
                state_dict[puzzle_emb_name] = new_weights
        model.load_state_dict(state_dict, assign=True)
def compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step,
        base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps,
        min_ratio=config.lr_min_ratio
    )



def create_evaluators(config: PretrainConfig, eval_metadata: PuzzleDatasetMetadata) -> List[Any]:
    data_paths = config.data_paths_test if len(config.data_paths_test)>0 else config.data_paths
    # Initialize evaluators
    #print('evaluator', data_paths, config.evaluators)
    evaluators = []
    for cfg in config.evaluators:
        for data_path in data_paths:
            klass = load_model_class(cfg.name, "evaluators.")
            #print('klass', klass)
            cls = klass(
                data_path=data_path, eval_metadata=eval_metadata, **cfg.__pydantic_extra__
            )  # type: ignore
            #print('cls', cls)
            evaluators.append(cls)

    return evaluators

def train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):
    train_state.step += 1
    if train_state.step > train_state.total_steps:  # At most train_total_steps
        return

    # To device
    batch = {k: v.cuda() for k, v in batch.items()}

    # Init carry if it is None
    if train_state.carry is None:
        with torch.device("cuda"):
            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore

    # Forward
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])

    ((1 / global_batch_size) * loss).backward()

    # Allreduce
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad)
            
    # Apply optimizer
    lr_this_step = None    
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)

        for param_group in optim.param_groups:
            param_group['lr'] = lr_this_step
            
        optim.step()
        optim.zero_grad()

    # Reduce metrics
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())

        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
        # Reduce and reconstruct
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1:
            dist.reduce(metric_values, dst=0)

        if rank == 0:
            metric_values = metric_values.cpu().numpy()
            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}
            
            # Postprocess
            count = max(reduced_metrics["count"], 1)  # Avoid NaNs
            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}

            reduced_metrics["train/lr"] = lr_this_step
            return reduced_metrics

def evaluate(
    config: PretrainConfig,
    train_state: TrainState,
    eval_loader: torch.utils.data.DataLoader,
    eval_metadata: PuzzleDatasetMetadata,
    evaluators: List[Any],
    rank: int,
    world_size: int,
    cpu_group: Optional[dist.ProcessGroup],
):
    reduced_metrics = None

    with torch.inference_mode():
        return_keys = set(config.eval_save_outputs)
        for evaluator in evaluators:
            evaluator.begin_eval()
            return_keys.update(evaluator.required_outputs)

        # Run evaluation
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}

        save_preds = {}

        metric_keys = []
        metric_values = None

        carry = None
        processed_batches = 0
        
        for set_name, batch, global_batch_size in eval_loader:
            processed_batches += 1
            if rank == 0:
                print(f"Processing batch {processed_batches}: {set_name}")
            
            # To device
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)  # type: ignore

            # Forward
            inference_steps = 0
            while True:
                carry, loss, metrics, preds, all_finish = train_state.model(
                    carry=carry, batch=batch, return_keys=return_keys
                )
                inference_steps += 1

                if all_finish:
                    break

            if rank == 0:
                print(f"  Completed inference in {inference_steps} steps")

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        save_preds.setdefault(k, [])
                        save_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory

            for evaluator in evaluators:
                evaluator.update_batch(batch, preds)

            del carry, loss, preds, batch, all_finish

            # Aggregate metrics
            set_id = set_ids[set_name]

            if metric_values is None:
                metric_keys = list(
                    sorted(metrics.keys())
                )  # Sort keys to guarantee all processes use the same order.
                metric_values = torch.zeros(
                    (len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda"
                )

            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])

            del metrics

        # concatenate save preds
        save_preds = {k: torch.cat(v, dim=0) for k, v in save_preds.items()}

        # Save preds
        if config.checkpoint_path is not None and len(save_preds):
            # Each rank save predictions independently
            os.makedirs(os.path.dirname(config.checkpoint_path), exist_ok=True)
            torch.save(
                save_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}")
            )

        del save_preds

        # Reduce to rank 0
        if metric_values is not None:
            if world_size > 1:
                dist.reduce(metric_values, dst=0)

            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {
                    set_name: {
                        metric_name: reduced_metrics[set_id, metric_id]
                        for metric_id, metric_name in enumerate(metric_keys)
                    }
                    for set_id, set_name in enumerate(set_ids)
                }

                # Postprocess
                for set_name, m in reduced_metrics.items():
                    count = m.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in m.items()}

        # Run evaluators
        if rank == 0:
            print(f"\nRunning {len(evaluators)} evaluator(s)...")
            
        for i, evaluator in enumerate(evaluators):
            if rank == 0:
                print(f"Running evaluator {i+1}/{len(evaluators)}: {evaluator.__class__.__name__}")
                
            # Path for saving
            evaluator_save_path = None
            if config.checkpoint_path is not None:
                evaluator_save_path = os.path.join(
                    config.checkpoint_path,
                    f"evaluator_{evaluator.__class__.__name__}_step_{train_state.step}",
                )
                os.makedirs(evaluator_save_path, exist_ok=True)

            # Run and log
            metrics = evaluator.result(evaluator_save_path, rank=rank, world_size=world_size, group=cpu_group)
            if rank == 0 and metrics is not None:
                if reduced_metrics is None:
                    reduced_metrics = {}

                reduced_metrics.update(metrics)
                print(f"  Completed {evaluator.__class__.__name__}")
                
        if rank == 0:
            print("All evaluators completed!")

    return reduced_metrics

def save_code_and_config(config: PretrainConfig):
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)

    # Copy code
    code_list = [
        get_model_source_path(config.arch.name),
        get_model_source_path(config.arch.loss.name)
    ]
    for code_file in code_list:
        if code_file is not None:
            code_name = os.path.basename(code_file)

            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))

    # Dump config as yaml
    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")
    with open(config_file, "wt") as f:
        yaml.dump(config.model_dump(), f)

    # Log code
    print(config.checkpoint_path)


def load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)  # type: ignore

        # Naming
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_paths[0]).capitalize()}-ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)

        objects = [config]

    if world_size > 1:
        dist.broadcast_object_list(objects, src=0)

    return objects[0]  # type: ignore


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config: DictConfig):
    RANK = 0
    WORLD_SIZE = 1
    CPU_PROCESS_GROUP = None

    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
        
        # CPU GLOO process group
        CPU_PROCESS_GROUP = dist.new_group(backend="gloo")
        assert (
            dist.get_rank(CPU_PROCESS_GROUP) == RANK and dist.get_world_size(CPU_PROCESS_GROUP) == WORLD_SIZE
        )

    # Load sync'ed config
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)

    # Seed RNGs to ensure consistency
    torch.random.manual_seed(config.seed + RANK)

    # Dataset
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter

    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."

    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    try:
        eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    except:
        print("NO EVAL DATA FOUND")
        eval_loader = eval_metadata = None

    try:
        evaluators = create_evaluators(config, eval_metadata)
    except:
        print("No evaluator found")
        evaluators = []

    # Train state
    train_state = init_train_state(config, train_metadata, rank=RANK, world_size=WORLD_SIZE)

    # Progress bar and logger
    progress_bar = None
    ema_helper = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)
        print({"num_params": sum(x.numel() for x in train_state.model.parameters())})
        save_code_and_config(config)
    if config.ema:
        print('Setup EMA')
        ema_helper = EMAHelper(mu=config.ema_rate)
        ema_helper.register(train_state.model)

    # Training Loop
    for _iter_id in range(total_iters):
        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")

        ############ Train Iter
        if RANK == 0:
            print("TRAIN")
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                #print(metrics, train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore
            if config.ema:
                ema_helper.update(train_state.model)

        if _iter_id >= config.min_eval_interval:
            ############ Evaluation
            if RANK == 0:
                print("EVALUATE")
            if config.ema:
                print("SWITCH TO EMA")
                train_state_eval = copy.deepcopy(train_state)
                train_state_eval.model = ema_helper.ema_copy(train_state_eval.model)
            else:
                train_state_eval = train_state
            train_state_eval.model.eval()
            metrics = evaluate(config, 
                train_state_eval, 
                eval_loader, 
                eval_metadata, 
                evaluators,
                rank=RANK, 
                world_size=WORLD_SIZE,
                cpu_group=CPU_PROCESS_GROUP)

            if RANK == 0 and metrics is not None:
                print(metrics, train_state.step)
                
            ############ Checkpointing
            if RANK == 0:
                print("SAVE CHECKPOINT")
            if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
                save_train_state(config, train_state_eval)

            if config.ema:
                del train_state_eval

    # finalize
    if dist.is_initialized():
        dist.destroy_process_group()



if __name__ == "__main__":
    launch()

Overwriting eval-arc.py


In [13]:
# ! mkdir data1
! cp /root/EdgeTRM/arc-prize-2025/arc-agi_test_challenges.json data1

In [14]:
# !ls /root/EdgeTRM/arc-prize-2024
!ls /root/EdgeTRM/arc-prize-2025

arc-agi_evaluation_challenges.json  arc-agi_training_solutions.json
arc-agi_evaluation_solutions.json   arc_2025_setup
arc-agi_test_challenges.json	    sample_submission.json
arc-agi_training_challenges.json


In [15]:
import os
import json

# # 1. Path to the 2024 competition data
# SOURCE_DIR = '/root/EdgeTRM/arc-prize-2024'
# WORKING_DIR = '/root/EdgeTRM/arc-prize-2024/arc_2024_setup'

# # # 1. Path to the 2025 competition data
SOURCE_DIR = '/root/EdgeTRM/arc-prize-2025'
WORKING_DIR = '/root/EdgeTRM/arc-prize-2025/arc_2025_setup'
os.makedirs(WORKING_DIR, exist_ok=True)

def setup_arc_2024(subset_name, has_solutions=True):
    challenge_src = os.path.join(SOURCE_DIR, f'arc-agi_{subset_name}_challenges.json')
    solution_src = os.path.join(SOURCE_DIR, f'arc-agi_{subset_name}_solutions.json')
    
    challenge_link = os.path.join(WORKING_DIR, f'arc_{subset_name}_challenges.json')
    solution_link = os.path.join(WORKING_DIR, f'arc_{subset_name}_solutions.json')

    # Link Challenges
    if os.path.lexists(challenge_link): os.remove(challenge_link)
    if os.path.exists(challenge_src):
        os.symlink(challenge_src, challenge_link)
    else:
        print(f"File not found: {challenge_src}")
        return

    # Handle Solutions
    if has_solutions and os.path.exists(solution_src):
        if os.path.lexists(solution_link): os.remove(solution_link)
        os.symlink(solution_src, solution_link)
        print(f"Linked real data for: {subset_name}")
    else:
        # Create dummy solutions as simple lists of grids for the builder
        with open(challenge_src, 'r') as f:
            challenges = json.load(f)
        dummy_data = {k: [[[0]] for _ in v['test']] for k, v in challenges.items()}
        with open(solution_link, 'w') as f:
            json.dump(dummy_data, f)
        print(f"Linked challenges and created dummy solutions for: {subset_name}")

# Training and Evaluation have real solutions in the 2024 folder
setup_arc_2024('training', has_solutions=True)
setup_arc_2024('evaluation', has_solutions=True)

# Test does not have a solutions file in the competition folder
setup_arc_2024('test', has_solutions=False)

print(f"\nUse this prefix for your build script: {WORKING_DIR}/arc")

Linked real data for: training
Linked real data for: evaluation
Linked challenges and created dummy solutions for: test

Use this prefix for your build script: /root/EdgeTRM/arc-prize-2025/arc_2025_setup/arc


In [16]:
# Generate the test puzzle dataset. The paper uses 1,000 augmentations for test-time evaluation.
# We default to 1,000 for true paper-aligned results, but you can set --num-aug 128 for a faster run.
! python -m dataset.build_arc_dataset \
  --input-file-prefix  /root/EdgeTRM/arc-prize-2025/arc_2025_setup/arc \
  --output-dir data1/arc2test-aug-1000 \
  --subsets test \
  --test-set-name test \
  --num-aug 1000

[Puzzle 22233c11] augmentation not full, only 576
[Puzzle 332efdb3] augmentation not full, only 9
[Puzzle 4258a5f9] augmentation not full, only 576
[Puzzle 0b17323b] augmentation not full, only 288
[Puzzle 1b60fb0c] augmentation not full, only 574
[Puzzle 18419cfa] augmentation not full, only 576
[Puzzle 2697da3f] augmentation not full, only 72
[Puzzle 3906de3d] augmentation not full, only 576
[Puzzle 1c0d0a4b] augmentation not full, only 576
[Puzzle 2281f1f4] augmentation not full, only 576
[Puzzle 31adaf00] augmentation not full, only 576
[Puzzle 239be575] augmentation not full, only 576
[Puzzle 017c7c7b] augmentation not full, only 576
[Puzzle 2c0b0aff] augmentation not full, only 576
[Puzzle 05f2a901] augmentation not full, only 576
[Puzzle 28e73c20] augmentation not full, only 72
[Puzzle 0d87d2a6] augmentation not full, only 576
[Puzzle 2faf500b] augmentation not full, only 576
[Puzzle 25ff71a9] augmentation not full, only 576
[Puzzle 253bf280] augmentation not full, only 576
[Puz

In [17]:
!pwd

/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [ ]:
import os
if 1 or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !HYDRA_FULL_ERROR=1 WANDB_MODE=disabled torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
    eval-arc.py \
    arch=trm \
    data_paths="[./data1/arc2test-aug-1000]" \
    arch.L_layers=2 \
    arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
    freeze_weights=False \
    +load_checkpoint=/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907 \
    +checkpoint_path=./eval_checkpoint \
    eval_interval=5000 \
    epochs=5000 \
    global_batch_size=512 \
    ema=True \
    lr_warmup_steps=1000 \
    lr=0.0001   
else:
    !HYDRA_FULL_ERROR=1 WANDB_MODE=disabled torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
    eval-arc.py \
    arch=trm \
    data_paths="[./data1/arc2test-aug-1000]" \
    arch.L_layers=2 \
    arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
    freeze_weights=False \
    +load_checkpoint=/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907 \
    +checkpoint_path=./eval_checkpoint \
    eval_interval=5000 \
    epochs=5000 \
    global_batch_size=512 \
    ema=True \
    lr_warmup_steps=1000 \
    lr=0.0001   

---
## Section 4 — Evaluation Setup

We now evaluate the checkpoint-loaded ARC model:
- **FP32**: baseline (as loaded)


In [ ]:
# diagnose_modal.py
# ─────────────────────────────────────────────────────────────────────────────
# This script isolates the model loading and high-performance puzzle-by-puzzle
# evaluation to verify that the FP32 baseline accuracy is successfully restored.
# Run this on Modal!
# ─────────────────────────────────────────────────────────────────────────────

import sys
import os
import torch
import numpy as np
import json

# Add repo to path
sys.path.append(os.getcwd())

# Ensure correct data dir and devices
DATA_DIR = "./data1/arc2test-aug-1000"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "step_723914"

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Data Dir: {DATA_DIR}")

# 1. Import model components
from trm import TinyRecursiveReasoningModel_ACTV1
from torch.utils.data import Dataset, DataLoader

class ARCDataset(Dataset):
    def __init__(self, split_dir: str):
        self.inputs = np.load(f"{split_dir}/all__inputs.npy")
        self.labels = np.load(f"{split_dir}/all__labels.npy")

        puzzle_ids  = np.load(f"{split_dir}/all__puzzle_identifiers.npy")
        puzzle_ptr  = np.load(f"{split_dir}/all__puzzle_indices.npy")

        counts = np.diff(puzzle_ptr).astype(np.int64)
        self.per_sample_pids = np.repeat(puzzle_ids, counts)

        assert len(self.inputs) == len(self.per_sample_pids), (
            f"Shape mismatch: inputs={len(self.inputs)}, pids={len(self.per_sample_pids)}"
        )

        with open(f"{split_dir}/../train/dataset.json") as fj:
            meta = json.load(fj)
        self.seq_len               = meta["seq_len"]
        self.vocab_size            = meta["vocab_size"]
        self.num_puzzle_identifiers = meta["num_puzzle_identifiers"]

    def __len__(self): return len(self.inputs)

    def __getitem__(self, i):
        return (
            torch.tensor(self.inputs[i],          dtype=torch.long),
            torch.tensor(self.labels[i],           dtype=torch.long),
            torch.tensor(self.per_sample_pids[i],  dtype=torch.long),
        )

# 2. Re-create the load_arc_model function with aligned H_cycles=3, L_cycles=4, halt_max_steps=16
config_data = """
arch:
  H_cycles: 3
  H_layers: 0
  L_cycles: 4
  L_layers: 2
  expansion: 4
  forward_dtype: bfloat16
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_len: 16
  puzzle_emb_ndim: 512
global_batch_size: 512
"""

# Re-implement notebook's cell 7 load_arc_model helper
import yaml

def load_arc_model(checkpoint_path, config_text):
    raw_config = yaml.safe_load(config_text)
    arch = raw_config['arch']
    
    final_config = {
        "batch_size": 32,
        "seq_len": 900,
        "num_puzzle_identifiers": 1191730,
        "vocab_size": 12,
        "H_cycles": arch['H_cycles'],
        "L_cycles": arch['L_cycles'],
        "H_layers": arch['H_layers'],
        "L_layers": arch['L_layers'],
        "hidden_size": arch['hidden_size'],
        "expansion": arch['expansion'],
        "num_heads": arch['num_heads'],
        "pos_encodings": arch['pos_encodings'],
        "halt_max_steps": arch['halt_max_steps'],
        "halt_exploration_prob": arch['halt_exploration_prob'],
        "forward_dtype": arch.get('forward_dtype', 'bfloat16'),
        "mlp_t": arch.get('mlp_t', False),
        "puzzle_emb_ndim": arch.get('puzzle_emb_ndim', 512),
        "puzzle_emb_len": arch.get('puzzle_emb_len', 16),
        "no_ACT_continue": arch.get('no_ACT_continue', True)
    }

    model = TinyRecursiveReasoningModel_ACTV1(config_dict=final_config)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            clean_state_dict[k[len(unwanted_prefix):]] = v
        else:
            clean_state_dict[k] = v
            
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = model.inner.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
            mean_emb = torch.mean(puzzle_emb, dim=0)
            new_weights[:] = mean_emb
            min_rows = min(puzzle_emb.shape[0], expected_shape[0])
            new_weights[:min_rows] = puzzle_emb[:min_rows]
            clean_state_dict[puzzle_emb_name] = new_weights
            
    model.load_state_dict(clean_state_dict)
    model.__dict__['model'] = model
    model.eval()
    print("Model loaded successfully!")
    return model

print("Loading model...")
model = load_arc_model(CHECKPOINT_PATH, config_data)

# Helper function to unwrap
def get_inner(m):
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2

# 3. Load ARC dataset
print("Loading dataset...")
test_ds = ARCDataset(f"{DATA_DIR}/test")
print(f"Test dataset has {len(test_ds)} samples.")

@torch.no_grad()
def evaluate_arc_per_puzzle(mdl, loader, device="cpu", n_sup_max=16, max_batches=None, return_pass2=False):
    from models.recursive_reasoning.trm import (
        TinyRecursiveReasoningModel_ACTV1Carry,
        TinyRecursiveReasoningModel_ACTV1InnerCarry,
    )
    from dataset.build_arc_dataset import inverse_aug, grid_hash, arc_grid_to_np, PuzzleIdSeparator
    from evaluators.arc import _crop
    import json
    import os
    import numpy as np
    import time
    from tqdm import tqdm

    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)

    # Load mappings
    with open(os.path.join(DATA_DIR, "identifiers.json"), "r") as f:
        identifier_map = json.load(f)
    with open(os.path.join(DATA_DIR, "test_puzzles.json"), "r") as f:
        test_puzzles = json.load(f)

    # High-performance caching of inverse_aug
    aug_cache = {}
    def get_aug(pid):
        if pid not in aug_cache:
            name = identifier_map[pid]
            aug_cache[pid] = inverse_aug(name)
        return aug_cache[pid]

    # High-performance caching of _crop to eliminate Numba overhead on repetitive sequences
    crop_cache = {}
    def get_crop(seq):
        seq_bytes = seq.tobytes()
        if seq_bytes not in crop_cache:
            crop_cache[seq_bytes] = _crop(seq)
        return crop_cache[seq_bytes]

    # Precompute canonical input hashes
    precomputed_input_info = {}
    ds = loader.dataset
    if hasattr(ds, "inputs") and hasattr(ds, "per_sample_pids"):
        try:
            puzzle_ids_arr = None
            puzzle_ptr_arr = None
            for split in ["test", "train"]:
                ids_path = os.path.join(DATA_DIR, split, "all__puzzle_identifiers.npy")
                ptr_path = os.path.join(DATA_DIR, split, "all__puzzle_indices.npy")
                if os.path.exists(ids_path):
                    ptr = np.load(ptr_path)
                    if ptr[-1] == len(ds):
                        puzzle_ids_arr = np.load(ids_path)
                        puzzle_ptr_arr = ptr
                        break
            if puzzle_ids_arr is not None and puzzle_ptr_arr is not None:
                puzzle_test_hashes = {
                    name: [grid_hash(arc_grid_to_np(pair["input"])) for pair in pz["test"]]
                    for name, pz in test_puzzles.items()
                }
                for j in range(len(puzzle_ids_arr)):
                    pid = int(puzzle_ids_arr[j])
                    if pid == 0: continue
                    name = identifier_map[pid]
                    orig_name = name.split(PuzzleIdSeparator)[0]
                    if orig_name not in test_puzzles: continue
                    E = len(test_puzzles[orig_name]["test"])
                    start_ptr = int(puzzle_ptr_arr[j])
                    end_ptr = int(puzzle_ptr_arr[j+1])
                    for k in range(start_ptr, end_ptr):
                        test_example_index = (k - start_ptr) % E
                        input_hash = puzzle_test_hashes[orig_name][test_example_index]
                        precomputed_input_info[k] = (orig_name, input_hash)
        except Exception as e:
            print(f"[WARN] Input hash precomputation failed, falling back: {e}")

    local_hmap = {}       # pred_hash -> canonical grid np.ndarray
    local_preds = {}      # orig_name -> {input_hash -> [(pred_hash, q_val), ...]}

    t0 = time.time()
    
    # Bypass DataLoader collation completely to eliminate sequential __getitem__ CPU tensor allocation overhead
    ds = loader.dataset
    inputs_np = ds.inputs
    labels_np = ds.labels
    pids_np = ds.per_sample_pids
    num_samples = len(inputs_np)
    batch_size = loader.batch_size if hasattr(loader, "batch_size") else 512
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    pbar = tqdm(range(num_batches), desc="Evaluating per-puzzle batches", leave=True)
    for batch_idx in pbar:
        if max_batches is not None and batch_idx >= max_batches:
            break

        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        x_batch = torch.from_numpy(inputs_np[start_idx:end_idx]).to(device, dtype=torch.long)
        y_true  = torch.from_numpy(labels_np[start_idx:end_idx]).to(device, dtype=torch.long)
        pids    = torch.from_numpy(pids_np[start_idx:end_idx]).to(device, dtype=torch.long)

        batch = {
            "inputs":             x_batch.to(torch.int32),
            "labels":             y_true.to(torch.int32),
            "puzzle_identifiers": pids.to(torch.int32),
        }

        carry = inner.initial_carry(batch)
        ic    = carry.inner_carry
        cast  = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )

        last_outputs = None
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            last_outputs = outputs
            if carry.halted.all():
                break

        if last_outputs is None:
            continue

        preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()  # (B, seq_len)
        q_logits    = last_outputs.get("q_halt_logits", torch.zeros(preds_batch.shape[0], device=device))
        q_values    = q_logits.sigmoid().cpu().numpy().flatten()    # (B,)

        inputs_cpu  = inputs_np[start_idx:end_idx]
        pids_cpu    = pids_np[start_idx:end_idx]

        for i in range(preds_batch.shape[0]):
            identifier = pids_cpu[i]
            if identifier == 0:  # Skip blank padding
                continue

            orig_name, _inverse_fn = get_aug(identifier)
            pred_seq = preds_batch[i]
            q_val = float(q_values[i])

            sample_idx = start_idx + i
            if sample_idx in precomputed_input_info and precomputed_input_info[sample_idx][0] == orig_name:
                input_hash = precomputed_input_info[sample_idx][1]
            else:
                inp_seq = inputs_cpu[i]
                input_grid = _inverse_fn(get_crop(inp_seq))
                input_hash = grid_hash(input_grid)

            # Crop and inverse transform prediction
            pred_grid = _inverse_fn(get_crop(pred_seq))
            pred_hash = grid_hash(pred_grid)

            local_hmap[pred_hash] = pred_grid

            local_preds.setdefault(orig_name, {})
            local_preds[orig_name].setdefault(input_hash, [])
            local_preds[orig_name][input_hash].append((pred_hash, q_val))

        # Update progress bar set_postfix on every single batch for immediate visual logging
        if True:
            run_evaluated = [name for name in test_puzzles.keys() if name in local_preds]
            if len(run_evaluated) > 0:
                run_correct = [0.0, 0.0]
                run_cell_hits = 0
                run_n_cells = 0
                for name in run_evaluated:
                    puzzle = test_puzzles[name]
                    num_correct = [0, 0]
                    for pair in puzzle["test"]:
                        inp_grid = arc_grid_to_np(pair["input"])
                        out_grid = arc_grid_to_np(pair["output"])
                        input_hash = grid_hash(inp_grid)
                        label_hash = grid_hash(out_grid)

                        p_map = {}
                        for h, q in local_preds[name].get(input_hash, []):
                            p_map.setdefault(h, [0, 0.0])
                            p_map[h][0] += 1
                            p_map[h][1] += q

                        if not len(p_map):
                            run_n_cells += out_grid.size
                            continue

                        for h, stats in p_map.items():
                            stats[1] /= stats[0]

                        p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)

                        for i, k in enumerate([1, 2]):
                            ok = False
                            for h, stats in p_map_sorted[:k]:
                                ok |= (h == label_hash)
                            num_correct[i] += int(ok)

                        top_hash = p_map_sorted[0][0]
                        top_grid = local_hmap[top_hash]
                        if top_grid.shape == out_grid.shape:
                            run_cell_hits += (top_grid == out_grid).sum()
                        run_n_cells += out_grid.size

                    for i in range(2):
                        run_correct[i] += num_correct[i] / len(puzzle["test"])

                run_p1 = run_correct[0] / len(run_evaluated)
                run_cell = run_cell_hits / run_n_cells if run_n_cells > 0 else 0.0
                pbar.set_postfix({"p1": f"{run_p1*100:.2f}%", "cell": f"{run_cell*100:.2f}%"})
                
                # Print directly to stdout on every single batch so it is permanently logged in Modal logs
                print(f" -> Batch {batch_idx + 1}/{num_batches} | Running Pass@1: {run_p1*100:.4f}% | Running Cell Acc: {run_cell*100:.4f}%", flush=True)

    elapsed = time.time() - t0

    # paper-compliant aggregated voting and accuracy evaluation
    pass_Ks = [1, 2, 5]
    correct = [0.0 for _ in pass_Ks]
    evaluated_puzzles = [name for name in test_puzzles.keys() if name in local_preds]
    n_puzzles = len(evaluated_puzzles)

    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        num_test_correct = [0 for _ in pass_Ks]

        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])

            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)

            p_map = {}
            for h, q in local_preds.get(name, {}).get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q

            if not len(p_map):
                continue

            for h, stats in p_map.items():
                stats[1] /= stats[0]

            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)

            for i, k in enumerate(pass_Ks):
                ok = False
                for h, stats in p_map_sorted[:k]:
                    ok |= (h == label_hash)
                num_test_correct[i] += int(ok)

        for i in range(len(pass_Ks)):
            correct[i] += num_test_correct[i] / len(puzzle["test"])

    # Cell accuracy estimation
    cell_hits = 0
    n_cells = 0
    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)

            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list:
                continue

            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            for h, stats in p_map.items():
                stats[1] /= stats[0]

            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            top_hash = p_map_sorted[0][0]
            top_grid = local_hmap[top_hash]

            if top_grid.shape == out_grid.shape:
                cell_hits += (top_grid == out_grid).sum()
                n_cells += out_grid.size
            else:
                n_cells += out_grid.size

    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    ms_per_puzzle = elapsed / n_puzzles if n_puzzles > 0 else 0.0

    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles

# 5. Run evaluation
print("Running FP32 baseline evaluation...")
# Configure DataLoader with highly optimized batch size for GPU
BATCH_SIZE = 2048
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

p1, p2, cell, ms, npuzz = evaluate_arc_per_puzzle(
    model, test_loader, device=DEVICE, n_sup_max=16, return_pass2=True
)
print("\n[Evaluation Complete]")
print(f"Pass@1 Exact: {p1*100:.2f}%")
print(f"Pass@2 Exact: {p2*100:.2f}%")
print(f"Cell Accuracy: {cell*100:.2f}%")
print(f"Latency: {ms:.2f} ms/puzzle")
print(f"Evaluated puzzles: {npuzz}")


Device: cuda


---
## Section 5 — Training a TRM Model from Scratch

In this section, we implement a premium-grade training pipeline to train a fresh `TinyRecursiveReasoningModel_ACTV1` model from scratch on the ARC-AGI dataset.

### Training Details & Hyperparameters
- **Main Optimizer**: `AdamAtan2` (Atan2-based gradient updates for superior learning) with fallback to `AdamW` if not installed.
- **Embedding Optimizer**: `CastedSparseEmbeddingSignSGD_Distributed` to update sparse puzzle embeddings using SignSGD.
- **Learning Rate Schedule**: Cosine learning rate decay with a linear warmup phase.
- **Model Configuration**: Paper-aligned architecture ($H_{cycles}=4, L_{cycles}=4, L_{layers}=2, hidden\_size=512$, vocab_size=12, seq_len=900, puzzle embedding dimension 512, halt steps 16).
- **VRAM Management**: Batch size of `256` prevents GPU memory swapping and PCIe bottlenecking, ensuring maximum local execution speed.

In [ ]:
# ── 5.1  Model & Dataset Initialization ──────────────────────────────────────
import os
import sys
import math
import copy
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
from tqdm.notebook import tqdm

# Add repo to path if needed
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
from models.losses import ACTLossHead
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed

# Attempt to load AdamAtan2 from paper, fallback to AdamW
try:
    from adam_atan2_pytorch import AdamAtan2
    print("Successfully imported AdamAtan2!")
except ModuleNotFoundError:
    from torch.optim import AdamW as AdamAtan2
    print("WARNING: adam_atan2_pytorch not found, using AdamW as fallback.")

# Hyperparameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data1/arc2test-aug-1000"
CHECKPOINT_DIR = "./checkpoints/trm_scratch"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 256  # VRAM-friendly batch size to prevent swapping/PCIe bottlenecks
LR = 1e-4
PUZZLE_EMB_LR = 1e-2
WEIGHT_DECAY = 0.1
PUZZLE_EMB_WEIGHT_DECAY = 0.1
TOTAL_STEPS = 5000
WARMUP_STEPS = 500
LR_MIN_RATIO = 0.1
CHECKPOINT_INTERVAL = 500
NUM_PUZZLE_IDENTIFIERS = 225972  # Vocabulary size matching the dataset size (including <blank>)

print(f"Device: {DEVICE}")
print(f"Initializing loaders from: {DATA_DIR}")

# 1. Build Datasets & Loaders
train_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=False,
    epochs_per_iter=1,
    rank=0,
    num_replicas=1
)
train_ds = PuzzleDataset(train_ds_config, split="train")

test_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=True,
    epochs_per_iter=1,
    rank=0,
    num_replicas=1
)
test_ds = PuzzleDataset(test_ds_config, split="test")

train_loader = DataLoader(train_ds, batch_size=None, num_workers=1, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=1, pin_memory=True)

print("Loading dataset metadata...")
metadata = train_ds.metadata
print(f"Vocab Size: {metadata.vocab_size}, Sequence Length: {metadata.seq_len}")
print(f"Total groups: {metadata.total_groups}, Total puzzles: {metadata.total_puzzles}")

# 2. Instantiate TRM Model with Loss Head
model_config = {
    "batch_size": BATCH_SIZE,
    "seq_len": metadata.seq_len,
    "puzzle_emb_ndim": 512,
    "num_puzzle_identifiers": NUM_PUZZLE_IDENTIFIERS,
    "vocab_size": metadata.vocab_size,
    "H_cycles": 4,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

print("Initializing fresh TinyRecursiveReasoningModel_ACTV1 model...")
base_model = TinyRecursiveReasoningModel_ACTV1(model_config)
model = ACTLossHead(base_model, loss_type="stablemax_cross_entropy")
model = model.to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# 3. Instantiate Optimizers
# Embedding SignSGD optimizer
emb_optimizer = CastedSparseEmbeddingSignSGD_Distributed(
    model.model.puzzle_emb.buffers(),
    world_size=1,
    lr=PUZZLE_EMB_LR,
    weight_decay=PUZZLE_EMB_WEIGHT_DECAY
)

# Model AdamAtan2 / AdamW optimizer
main_optimizer = AdamAtan2(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95)
)

print("Model, datasets, and optimizers successfully initialized!")

In [ ]:
# ── 5.2  Training & Validation Loop ──────────────────────────────────────────
def get_lr_factor(step, total_steps, warmup_steps, min_ratio):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return min_ratio + max(0.0, (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress)))

print(f"Starting training from scratch for {TOTAL_STEPS} steps...")
model.train()

train_iter = iter(train_loader)
running_loss = 0.0
running_acc = 0.0
running_em = 0.0
running_steps = 0.0
log_window = 100

pbar = tqdm(range(1, TOTAL_STEPS + 1), desc="Training steps")
for step in pbar:
    try:
        set_name, batch, eff_batch_size = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        set_name, batch, eff_batch_size = next(train_iter)
        
    # Scale learning rates according to scheduler
    lr_scale = get_lr_factor(step, TOTAL_STEPS, WARMUP_STEPS, LR_MIN_RATIO)
    for param_group in main_optimizer.param_groups:
        param_group['lr'] = LR * lr_scale
    for param_group in emb_optimizer.param_groups:
        param_group['lr'] = PUZZLE_EMB_LR * lr_scale
        
    # Move batch to device
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    
    # Forward pass
    carry = model.initial_carry(batch)
    carry, loss, metrics, _, _ = model(carry=carry, batch=batch, return_keys=[])
    
    # Backward pass & Optimize
    loss_normalized = loss / BATCH_SIZE
    loss_normalized.backward()
    
    # Apply steps & Zero grads
    main_optimizer.step()
    main_optimizer.zero_grad()
    
    emb_optimizer.step()
    emb_optimizer.zero_grad()
    
    # Update metrics
    count = max(float(metrics.get("count", BATCH_SIZE)), 1.0)
    step_loss = float(loss.item()) / BATCH_SIZE
    step_acc = float(metrics.get("accuracy", 0.0)) / count
    step_em = float(metrics.get("exact_accuracy", 0.0)) / count
    step_steps = float(metrics.get("steps", 0.0)) / count
    
    running_loss += (step_loss - running_loss) / min(step, log_window)
    running_acc += (step_acc - running_acc) / min(step, log_window)
    running_em += (step_em - running_em) / min(step, log_window)
    running_steps += (step_steps - running_steps) / min(step, log_window)
    
    pbar.set_postfix({
        "loss": f"{running_loss:.4f}",
        "acc": f"{running_acc*100:.2f}%",
        "em": f"{running_em*100:.2f}%",
        "steps": f"{running_steps:.1f}"
    })
    
    # Periodic evaluation and checkpointing
    if step % CHECKPOINT_INTERVAL == 0:
        # Save checkpoint
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"step_{step}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\n[Step {step}] Checkpoint saved to: {checkpoint_path}")
        
        # Run fast validation evaluation
        print(f"[Step {step}] Running fast per-puzzle evaluation...")
        p1, cell, ms, npuzz = evaluate_arc_per_puzzle(
            model, test_loader, device=DEVICE, n_sup_max=16, return_pass2=False
        )
        print(f"Validation Results -> Pass@1: {p1*100:.2f}% | Cell Acc: {cell*100:.2f}% | Latency: {ms:.2f} ms/puzzle")
        model.train()  # Restore training mode